In [1]:
%%sql
CREATE OR REPLACE TABLE gold.kpi_waste_summary AS
SELECT
    procurement_month,
    ROUND(SUM(waste_value_usd), 2)                              AS total_waste_value_usd,
    ROUND(SUM(CASE WHEN is_expired = true 
              THEN unit_cost_usd * qty_remaining ELSE 0 END), 2) AS expired_stock_value_usd,
    COUNT(CASE WHEN is_expired = true THEN 1 END)               AS expired_item_count,
    COUNT(CASE WHEN days_until_expiry BETWEEN 0 AND 10 
               THEN 1 END)                                      AS critical_zone_count,
    COUNT(CASE WHEN days_until_expiry BETWEEN 11 AND 30 
               THEN 1 END)                                      AS at_risk_count,
    ROUND(SUM(unit_cost_usd), 2)                                AS total_inventory_value_usd
FROM silver.fact_supply_inventory
GROUP BY procurement_month
ORDER BY procurement_month;

StatementMeta(, 33f467ab-fe06-4d8d-a5a2-e6d4f6a83b1b, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
CREATE OR REPLACE TABLE gold.kpi_waste_by_location AS
SELECT
    h.hospital_name,
    w.ward,
    ROUND(SUM(f.waste_value_usd), 2)                        AS total_waste_usd,
    COUNT(CASE WHEN f.is_expired = true THEN 1 END)         AS expired_count,
    COUNT(CASE WHEN f.days_until_expiry BETWEEN 0 
               AND 30 THEN 1 END)                           AS at_risk_count,
    SUM(f.qty_remaining)                                    AS total_qty_remaining
FROM silver.fact_supply_inventory f
JOIN silver.dim_hospital h ON f.hospital_id = h.hospital_id
JOIN silver.dim_ward w     ON f.ward_id = w.ward_id
GROUP BY h.hospital_name, w.ward
ORDER BY total_waste_usd DESC;

StatementMeta(, 33f467ab-fe06-4d8d-a5a2-e6d4f6a83b1b, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql
CREATE OR REPLACE TABLE gold.kpi_supplier_item AS
SELECT
    s.supplier_name,
    i.item_name,
    c.category,
    ROUND(SUM(f.waste_value_usd), 2)                        AS total_waste_usd,
    COUNT(CASE WHEN f.is_expired = true THEN 1 END)         AS expired_count,
    SUM(f.qty_remaining)                                    AS total_qty_remaining,
    ROUND(SUM(f.unit_cost_usd), 2)                          AS total_cost_usd,
    COUNT(CASE WHEN f.days_until_expiry 
               BETWEEN 0 AND 10 THEN 1 END)                 AS critical_zone_count
FROM silver.fact_supply_inventory f
JOIN silver.dim_supplier s  ON f.supplier_id = s.supplier_id
JOIN silver.dim_item i      ON f.item_id = i.item_id
JOIN silver.dim_category c  ON f.category_id = c.category_id
GROUP BY s.supplier_name, i.item_name, c.category
ORDER BY total_waste_usd DESC;

StatementMeta(, 33f467ab-fe06-4d8d-a5a2-e6d4f6a83b1b, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
SELECT 'kpi_waste_summary'     AS table_name, COUNT(*) AS row_count 
FROM gold.kpi_waste_summary
UNION ALL
SELECT 'kpi_waste_by_location', COUNT(*) 
FROM gold.kpi_waste_by_location
UNION ALL
SELECT 'kpi_supplier_item',     COUNT(*) 
FROM gold.kpi_supplier_item;

StatementMeta(, 33f467ab-fe06-4d8d-a5a2-e6d4f6a83b1b, 5, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [3]:
select * from gold.kpi_waste_by_location;

select hospital_name from gold.kpi_waste_by_location
where total_waste_usd = (select max(total_waste_usd) from gold.kpi_waste_by_location);

select distinct count(hospital_name) from gold.kpi_waste_by_location;




StatementMeta(, bdd3add0-3614-47ae-ab57-f9264f231c1f, 9, Finished, Available, Finished, True)

<Spark SQL result set with 35 rows and 6 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
SELECT 
    MIN(total_waste_usd) AS min_val,
    PERCENTILE_CONT(0.5) WITHIN GROUP 
        (ORDER BY total_waste_usd) AS mid_val,
    MAX(total_waste_usd) AS max_val
FROM gold.kpi_waste_by_location;

StatementMeta(, edce6d69-404a-4e28-9f1e-ba5ae5163bce, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>